# 4.4 — Temporal and Seasonal Variability

Diurnal and monthly MAE at Δ ≈ 3 h, MR=0.
Uses the extended aggregation (per-TOD, per-month) cached by common.py.

In [ ]:
import os, sys, datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
for _cand in (os.getcwd(),
              os.path.join(os.getcwd(), "notebooks", "analysis"),
              os.path.dirname(os.path.abspath("__file__"))):
    if os.path.isfile(os.path.join(_cand, "common.py")):
        if _cand not in sys.path: sys.path.insert(0, _cand)
        break
import importlib, common as C; importlib.reload(C)
plt.style.use("default")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "figure.facecolor": "white", "axes.facecolor": "white",
                     "savefig.facecolor": "white", "axes.edgecolor": "0.3",
                     "axes.labelcolor": "black", "xtick.color": "0.3",
                     "ytick.color": "0.3", "text.color": "black"})

RUNS  = C.discovered_runs()
MR0_RUNS = [r for r in RUNS if "mr0.00" in RUNS[r]]
ns = C.norm_stats(); VARS = ns["var_names"]; STD = ns["std"]
stn = C.station_table(); KEEP = C.keep_mask(stn, VARS)
AGG  = {r: C.load_agg(r, "mr0.00") for r in MR0_RUNS}
GRID = AGG[MR0_RUNS[0]]["grid"]; LEAD = C.lead_labels(GRID); K = len(GRID)
NV = len(VARS)
print("Models at MR=0.00:", MR0_RUNS)

# Extended aggregation — per-TOD and per-month
EXT = {r: C.load_ext(r, "mr0.00") for r in MR0_RUNS}
REF_KI = 6  # ≈ 3 h
print(f"Reference lead: {LEAD[REF_KI]}")
print("Extended caches loaded.")

## Diurnal MAE at Δ ≈ 3 h — by season

MAE at each 2-hour forecast-origin window (0–2 h, 2–4 h, …, 22–24 h UTC),
split by season. One row per season, one column per variable.

In [ ]:
# ── Compute 2-hourly × season cross-tabulation ───────────────────────────────
import torch, datetime

SEASON_LABELS = ["DJF", "MAM", "JJA", "SON"]
N_HBINS = 12  # 2-hour bins: [0,2), [2,4), ..., [22,24)
HBIN_LABELS = [f"{2*i}–{2*i+2}" for i in range(N_HBINS)]
HBIN_CENTRES = np.arange(1, 24, 2)  # 1, 3, 5, ..., 23

def _hbin_of(hours):
    """Hours-since-epoch -> 2-hour bin index 0..11."""
    return (hours % 24 / 2).astype(int)

def compute_hbin_season(run, mr="mr0.00"):
    """Return (4_sea, 12_hbin, K, V) model MAE and persistence MAE."""
    ns_ = C.norm_stats()
    VAR_ = ns_["var_names"]; STD_ = ns_["std"]
    stn_ = C.station_table(); KEEP_ = C.keep_mask(stn_, VAR_)
    raw = C.raw_test_obs()
    OBS, OMSK, H0, NT = raw["obs"], raw["mask"], raw["h0"], raw["nt"]
    d = C.load_dump(run, mr)
    import gc
    P, T, M = d["preds"], d["targets"], d["masks"]
    TH = d["target_hours"]
    Mw, K_, N_, _ = P.shape; NV_ = len(VAR_)

    mod_sum = np.zeros((4, N_HBINS, K_, NV_))
    mod_cnt = np.zeros((4, N_HBINS, K_, NV_))
    per_sum = np.zeros((4, N_HBINS, K_, NV_))
    per_cnt = np.zeros((4, N_HBINS, K_, NV_))
    clim_sum = np.zeros((4, N_HBINS, K_, NV_))
    clim_cnt = np.zeros((4, N_HBINS, K_, NV_))

    CLIM_LAG_STEPS = C.CLIM_LAG_STEPS  # 24 h on the 10-min raw-observation grid (144 steps)
    chunk = 256
    hours_to_row = C.hours_to_row  # raw_test_obs() is on the 10-min grid (6 rows per hour)
    for a0 in range(0, Mw, chunk):
        b0 = min(a0 + chunk, Mw)
        p = P[a0:b0].numpy().astype(np.float64)
        t = T[a0:b0, :, :, :NV_].numpy().astype(np.float64)
        m = (M[a0:b0, :, :, :NV_].numpy() > 0.5) & KEEP_[None, None]
        th = TH[a0:b0].numpy()
        t0h = th[:, 0]
        hbin = _hbin_of(t0h)
        sea = C._season_of(t0h)

        e_phys = np.abs((p - t) * STD_[None, None])

        ti_tgt  = hours_to_row(th, H0)
        ti_pers = hours_to_row(th[:, 0:1], H0)
        ok_t = (ti_tgt >= 0) & (ti_tgt < NT)
        ok_p = (ti_pers >= 0) & (ti_pers < NT)
        truth  = OBS[np.clip(ti_tgt, 0, NT-1)].astype(np.float64)
        tmask  = OMSK[np.clip(ti_tgt, 0, NT-1)] & ok_t[:, :, None, None]
        pers_v = OBS[np.clip(ti_pers, 0, NT-1)].astype(np.float64)
        pmask  = OMSK[np.clip(ti_pers, 0, NT-1)] & ok_p[:, :, None, None]
        per_ep = np.abs(pers_v - truth)
        mod_mk = m & tmask
        per_mk = m & tmask & pmask
        clim_v = OBS[np.clip(ti_tgt - CLIM_LAG_STEPS, 0, NT-1)].astype(np.float64)
        ti_clim = ti_tgt - CLIM_LAG_STEPS
        ok_c = (ti_clim >= 0) & (ti_clim < NT)
        cmask = OMSK[np.clip(ti_clim, 0, NT-1)] & ok_c[:, :, None, None]
        clim_ep = np.abs(clim_v - truth)
        clim_mk = m & tmask & cmask

        for si in range(4):
            for hi in range(N_HBINS):
                sel = (sea == si) & (hbin == hi)
                if not sel.any():
                    continue
                mod_sum[si, hi] += (e_phys[sel] * mod_mk[sel]).sum(axis=(0, 2))
                mod_cnt[si, hi] += mod_mk[sel].sum(axis=(0, 2))
                per_sum[si, hi] += (per_ep[sel] * per_mk[sel]).sum(axis=(0, 2))
                per_cnt[si, hi] += per_mk[sel].sum(axis=(0, 2))
                clim_sum[si, hi] += (clim_ep[sel] * clim_mk[sel]).sum(axis=(0, 2))
                clim_cnt[si, hi] += clim_mk[sel].sum(axis=(0, 2))
    mod_mae = np.where(mod_cnt > 0, mod_sum / mod_cnt, np.nan)
    per_mae = np.where(per_cnt > 0, per_sum / per_cnt, np.nan)
    clim_mae = np.where(clim_cnt > 0, clim_sum / clim_cnt, np.nan)
    del P, T, M, d
    gc.collect()
    return mod_mae, per_mae, clim_mae  # (4_sea, 12_hbin, K, V)

print("Computing 2-hourly × season cross-tab …")
HS_MOD, HS_PER, HS_CLIM = {}, {}, {}
for r in MR0_RUNS:
    print(f"  {r}…", end=" ", flush=True)
    HS_MOD[r], HS_PER[r], HS_CLIM[r] = compute_hbin_season(r)
    print("done")
print("All models computed.")


In [ ]:
# ── Figure: Diurnal MAE at Δ ≈ 3 h, 2-hourly, by season ────────────────────
fig, axes = plt.subplots(4, NV, figsize=(17, 13), sharex=True)

for si, sea_lbl in enumerate(SEASON_LABELS):
    for vi, v in enumerate(VARS):
        ax = axes[si, vi]
        for r in MR0_RUNS:
            label, col, _ = C.MODELS[r]
            ax.plot(HBIN_CENTRES, HS_MOD[r][si, :, REF_KI, vi],
                    "o-", ms=3, lw=1.3, color=col, label=label)
        # persistence
        ax.plot(HBIN_CENTRES, HS_PER[MR0_RUNS[0]][si, :, REF_KI, vi],
                "--", lw=1.1, color=C.BASELINE_COLORS["persistence"],
                label=C.BASELINE_LABELS["persistence"])
        ax.grid(alpha=0.3)
        if si == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if si == 3:
            ax.set_xlabel("Forecast-origin hour (UTC)", fontsize=8)
            ax.set_xticks(HBIN_CENTRES)
            ax.set_xticklabels(HBIN_LABELS, rotation=45, ha="right", fontsize=7)
    axes[si, 0].set_ylabel(f"{sea_lbl}\nMAE at {LEAD[REF_KI]}", fontsize=9)

# ── Sync y-axis per variable column ──────────────────────────────────────────
for vi in range(NV):
    col_axes = [axes[si, vi] for si in range(4)]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)
axes[0, -1].legend(fontsize=5.5, loc="upper left")
fig.suptitle(f"MAE at {LEAD[REF_KI]} by 2-hour forecast-origin bin (UTC) and season, all stations visible (MR=0); dashed = last-value persistence",
             y=1.01, fontsize=12)
plt.tight_layout()
C.save_fig(fig, "44_diurnal_hourly_by_season")
plt.show()
plt.close(fig)


## Monthly MAE at Δ ≈ 3 h — selected models

In [ ]:
SEL = ["v27", "v31", "lstm-baseline-v1"]
fig, axes = plt.subplots(1, NV, figsize=(17, 3.4))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    for r in SEL:
        e = EXT[r]
        label, col, _ = C.MODELS[r]
        mc = e["month_mod_cnt"][:, REF_KI, vi]
        ms_ = e["month_mod_sum"][:, REF_KI, vi]
        y = np.where(mc > 0, ms_ / np.maximum(mc, 1), np.nan)
        ax.plot(range(12), y, "o-", ms=4, lw=1.3, color=col, label=label)
    # persistence
    e0 = EXT[SEL[0]]
    pc = e0["month_per_cnt"][:, REF_KI, vi]
    ps = e0["month_per_sum"][:, REF_KI, vi]
    yp = np.where(pc > 0, ps / np.maximum(pc, 1), np.nan)
    ax.plot(range(12), yp, "--", lw=1.2, color=C.BASELINE_COLORS["persistence"],
            label=C.BASELINE_LABELS["persistence"])
    ax.set_xticks(range(12))
    ax.set_xticklabels(C.MONTH_LABELS, rotation=45, ha="right", fontsize=7)
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10); ax.grid(alpha=.3)
axes[0].set_ylabel(f"MAE at {LEAD[REF_KI]}")
axes[-1].legend(fontsize=7)
fig.suptitle(f"Monthly MAE at {LEAD[REF_KI]} (MR=0)", y=1.04)
plt.tight_layout(); C.save_fig(fig, "44_monthly"); plt.show()
plt.close(fig)

---
## Supplementary: diurnal MAE vs lead — full curves per TOD window

In [ ]:
for ti, tod_lbl in enumerate(C.TOD_LABELS):
    fig, axes = plt.subplots(1, NV, figsize=(17, 3.3))
    for vi, (ax, v) in enumerate(zip(axes, VARS)):
        for r in MR0_RUNS:
            label, col, _ = C.MODELS[r]
            y = C.pool_stations(EXT[r]["tod_mod_sum"][ti],
                                EXT[r]["tod_mod_cnt"][ti])[:, vi]
            ax.plot(range(1, K), y[1:], "o-", ms=3, lw=1.3, color=col, label=label)
        e0 = EXT[MR0_RUNS[0]]
        y_per  = C.pool_stations(e0["tod_per_sum"][ti], e0["tod_per_cnt"][ti])[:, vi]
        y_clim = C.pool_stations(e0["tod_clim_sum"][ti], e0["tod_clim_cnt"][ti])[:, vi]
        ax.plot(range(1, K), y_per[1:], "--", lw=1.2,
                color=C.BASELINE_COLORS["persistence"],
                label=C.BASELINE_LABELS["persistence"])
        ax.plot(range(1, K), y_clim[1:], "--", lw=1.2,
                color=C.BASELINE_COLORS["clim"],
                label=C.BASELINE_LABELS["clim"])
        ax.set_xticks(range(1, K, 2))
        ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6.5)
        ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10); ax.grid(alpha=.3)
    axes[0].set_ylabel("MAE [phys]")
    axes[-1].legend(fontsize=6, loc="upper left")
    fig.suptitle(f"Supplementary: MAE vs lead — {tod_lbl} (MR=0)", y=1.04)
    plt.tight_layout(); C.save_fig(fig, f"44_sup_tod_{ti}"); plt.show()
plt.close(fig)

## Interpretation

**Diurnal:** Wind MAE peaks during afternoon convection (12–18 UTC).
Temperature shows the smallest diurnal variation (strong spatial
coherence damps the TOD signal). If the model's diurnal profile
differs from persistence, it has learned time-of-day–dependent dynamics.

**Monthly:** Winter months (DJF) may show elevated temperature/pressure
errors (inversions, fog); summer months may show elevated wind errors
(convective instability).